# CNN: Forward Propagation From Scratch (Shared vs Non-Shared)

Notebook ini memuat model CNN terbaik dari notebook 1, menjalankan forward propagation scratch untuk arsitektur shared Conv2D, lalu menyediakan eksperimen non-shared parameter dengan LocallyConnected2D jika layer tersebut tersedia di TensorFlow/Keras.

## Setup

In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.wajib.cnn.models import CNN, trainModel, scratchFromKeras
from src.wajib.cnn.utils import loadBatch, loadDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_ROOT = PROJECT_ROOT / "data"
TRAIN_DIR = DATA_ROOT / "seg_train" / "seg_train"
TEST_DIR = DATA_ROOT / "seg_test" / "seg_test"
WEIGHTS_DIR = PROJECT_ROOT / "src" / "wajib" / "weights" / "cnn"
RESULTS_JSON = WEIGHTS_DIR / "cnn_experiment_results.json"
CLASS_NAMES_JSON = WEIGHTS_DIR / "class_names.json"

TARGET_SIZE = (64, 64)
VALIDATION_SIZE = 0.2
MAX_TEST_SAMPLES = 120  
TRAIN_NON_SHARED = True
NON_SHARED_EPOCHS = 5
BATCH_SIZE = 32


## Load Best Model

In [3]:
with open(RESULTS_JSON, "r", encoding="utf-8") as f:
    results = json.load(f)

best = max(results, key=lambda item: item["metrics"]["macro_f1"])
best_path = best["weights_path"]
best_config = best["config"]

with open(CLASS_NAMES_JSON, "r", encoding="utf-8") as f:
    class_names = json.load(f)

print("Best model:", best["name"])
print("Best path:", best_path)
print("Config:", best_config)


Best model: cnn_conv_L2_F32-64_K3-3_Pmax
Best path: /home/stahlynx/Coding/Semester-6/ML/Tejumama_Image-Captioning/src/wajib/weights/cnn/cnn_conv_L2_F32-64_K3-3_Pmax.keras
Config: {'num_conv_layers': 2, 'num_filters': [32, 64], 'filter_sizes': [3, 3], 'pooling_type': 'max'}


## Load test set

In [ ]:
# 3. Load test set untuk evaluasi scratch
train_paths, train_labels, _ = loadDataset(str(TRAIN_DIR), class_names=class_names)
test_paths, test_labels, _ = loadDataset(str(TEST_DIR), class_names=class_names)

if MAX_TEST_SAMPLES is not None:
    test_paths = test_paths[:MAX_TEST_SAMPLES]
    test_labels = test_labels[:MAX_TEST_SAMPLES]

X_test = loadBatch(test_paths, target_size=TARGET_SIZE)
y_test = np.asarray(test_labels, dtype=np.int64)

print(f"Scratch evaluation samples: {len(X_test)}")


Scratch evaluation samples: 120


## Load best keras model to CNNScratch shared param

In [ ]:
keras_model = tf.keras.models.load_model(best_path)
keras_model.summary()

scratch_model = scratchFromKeras(keras_model)
print("Scratch layers:", [layer.__class__.__name__ for layer in scratch_model.layers])
print("Keras params:", keras_model.count_params())
print("Scratch loaded params:", scratch_model.count_params())


Model: "cnn_conv_L2_F32-64_K3-3_Pmax"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,352,340 (24.23 MB)

 Trainable params: 2,117,446 (8.08 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,234,894 (16.15 MB)

Scratch layers: ['Conv2D', 'MaxPooling2D', 'Conv2D', 'MaxPooling2D', 'Flatten', 'Dense', 'Dense']
Keras params: 2117446
Scratch loaded params: 2117446


## Perbandingan prediksi Keras vs scratch shared Conv2D

In [6]:
keras_probs = keras_model.predict(X_test, verbose=0)
keras_pred = np.argmax(keras_probs, axis=1)

scratch_probs = scratch_model.predict(X_test)
scratch_pred = np.argmax(scratch_probs, axis=1)

comparison = pd.DataFrame([
    {
        "model": "Keras Conv2D shared",
        "accuracy": accuracy_score(y_test, keras_pred),
        "macro_f1": f1_score(y_test, keras_pred, average="macro"),
        "params": keras_model.count_params(),
    },
    {
        "model": "Scratch Conv2D shared",
        "accuracy": accuracy_score(y_test, scratch_pred),
        "macro_f1": f1_score(y_test, scratch_pred, average="macro"),
        "params": scratch_model.count_params(),
    },
])

display(comparison)
print("Max probability difference:", np.max(np.abs(keras_probs - scratch_probs)))


,model,accuracy,macro_f1,params
0,Keras Conv2D shared,0.683333,0.135314,2117446
1,Scratch Conv2D shared,0.683333,0.135314,2117446


Max probability difference: 1.1920929e-06


## Eksperimen non-shared parameter dengan LocallyConnected2D

In [7]:
non_shared_result = None

if TRAIN_NON_SHARED:
    try:
        train_paths, val_paths, y_train, y_val = train_test_split(
            train_paths,
            train_labels,
            test_size=VALIDATION_SIZE,
            random_state=SEED,
            stratify=train_labels,
        )
        X_train = __loadbatch__(train_paths, target_size=TARGET_SIZE)
        X_val = __loadbatch__(val_paths, target_size=TARGET_SIZE)
        y_train = np.asarray(y_train, dtype=np.int64)
        y_val = np.asarray(y_val, dtype=np.int64)

        non_shared_model = CNN(
            input_shape=X_train.shape[1:],
            num_classes=len(class_names),
            num_conv_layers=best_config["num_conv_layers"],
            num_filters=best_config["num_filters"],
            filter_sizes=best_config["filter_sizes"],
            pooling_type=best_config["pooling_type"],
            use_locally_connected=True,
        )
        non_shared_path = WEIGHTS_DIR / f"{best['name']}_non_shared.keras"
        non_shared_history = __train__(
            non_shared_model,
            X_train,
            y_train,
            X_val,
            y_val,
            epochs=NON_SHARED_EPOCHS,
            batch_size=BATCH_SIZE,
            save_path=str(non_shared_path),
        )
        non_shared_probs = non_shared_model.predict(X_test, verbose=0)
        non_shared_pred = np.argmax(non_shared_probs, axis=1)
        non_shared_result = {
            "model": "Keras LocallyConnected2D non-shared",
            "accuracy": accuracy_score(y_test, non_shared_pred),
            "macro_f1": f1_score(y_test, non_shared_pred, average="macro"),
            "params": non_shared_model.count_params(),
            "weights_path": str(non_shared_path),
        }
    except Exception as exc:
        print("Non-shared experiment skipped:", exc)
else:
    print("Non-shared experiment disabled. Set TRAIN_NON_SHARED = True to run it.")

if non_shared_result is not None:
    display(pd.concat([comparison, pd.DataFrame([non_shared_result])], ignore_index=True))


Non-shared experiment skipped: tf.keras.layers.LocallyConnected2D is not available in this TensorFlow/Keras version.


## Contoh prediksi per gambar


In [8]:
preview = pd.DataFrame({
    "path": [Path(path).name for path in test_paths[:20]],
    "true": [class_names[idx] for idx in y_test[:20]],
    "keras_shared": [class_names[idx] for idx in keras_pred[:20]],
    "scratch_shared": [class_names[idx] for idx in scratch_pred[:20]],
})
display(preview)


,path,true,keras_shared,scratch_shared
0,20057.jpg,buildings,buildings,buildings
1,20060.jpg,buildings,street,street
2,20061.jpg,buildings,buildings,buildings
3,20064.jpg,buildings,sea,sea
4,20073.jpg,buildings,buildings,buildings
5,20074.jpg,buildings,mountain,mountain
6,20078.jpg,buildings,buildings,buildings
7,20083.jpg,buildings,street,street
8,20094.jpg,buildings,buildings,buildings
9,20096.jpg,buildings,buildings,buildings
